
# Mehrgitter für singulär gestörte Probleme

Hier betrachten wir die Gleichung

## Initialisierung
Zunächst starten wir wie gehabt:


In [1]:
-- Initialize UG.
InitUG(2, AlgebraType("CPU", 1));

-- Load convenience scripts.
ug_load_script("../modsim2/toolbox.lua")

history = {}

* Initializing: paths... done, bridge... done, plugins... done                 *


In [2]:
configDesc = {
    geometry = {
        numRefs = 6, 
        gridName = "../grids/laplace_sample_grid_2d-tri.ugx", -- grid name
        requiredSubsets ={"Inner", "Boundary"},
    },
    
    order = nil, -- Lex
   
    problem = {
        id = "PoissonProblem",
        parameters = {
            epsx = 1.0,
            epsy = 1.0, 
        }   
    }
}

-- Create discretization.
domainDisc, approxSpace = util.modsim2.CreateDomainDisc(configDesc)

OrderLex(approxSpace, "x")    -- order vertices (and dofs) lexicographically

Loading Domain ../grids/laplace_sample_grid_2d-tri.ugx ... done.
Performing integrity check on domain ... done.

util.refinement: - refining level 0
util.refinement: - refining level 1
util.refinement: - refining level 2
| ---------------------------------------------------------------------------- |
|  Number of DoFs (All Procs)                                                  |
|  Algebra: Block 1 (divide by 1 for #Index)                                   |
|                                                                              |
|    GridLevel   |       Domain |     0: Inner |  1: Boundary                  |
| ---------------------------------------------------------------------------- |
| (lev,    0)    |            9 |            1 |            8 |
| (lev,    1)    |           25 |            9 |           16 |
| (lev,    2)    |           81 |           49 |           32 |
| (lev,    3)    |          289 |          225 |           64 |
| (lev,    0, g) |            9 |    

## Gauß-Seidel-Verfahren

In [3]:
-- solve problem with max 50 iterations, 
-- reducing the error by 6 orders of magnitude  
convCheck = ConvCheck()
convCheck:set_maximum_steps(50)
convCheck:set_reduction(1e-6)
convCheck:set_minimum_defect(1e-15)

solver = LinearSolver()
solver:set_preconditioner(GaussSeidel())
solver:set_convergence_check(convCheck)

dbgWriter = GridFunctionDebugWriter(approxSpace)
solver:set_debug(dbgWriter)

In [4]:
solver:set_preconditioner(GaussSeidel())
util.modsim2.SolverTest(domainDisc, approxSpace, solver, "SmootherError_GS.vtk");


   % %%%%%%%%  Iterative Linear Solver   %%%%%%%%%%%%%%%%%%
   % %%%%%%%%   (Precond: Gauss-Seidel)  %%%%%%%%%%%%%%%%%%
   %   Iter      Defect         Rate 
   %    0:    3.850905e+01      -------
   %    1:    1.163137e+01    3.020426e-01
   %    2:    4.059539e+00    3.490163e-01
   %    3:    1.655882e+00    4.078991e-01
   %    4:    8.286700e-01    5.004401e-01
   %    5:    5.019253e-01    6.056998e-01
   %    6:    3.428425e-01    6.830548e-01
   %    7:    2.504402e-01    7.304819e-01
   %    8:    1.909635e-01    7.625111e-01
   %    9:    1.501724e-01    7.863935e-01
   %   10:    1.209698e-01    8.055392e-01
   %   11:    9.946737e-02    8.222498e-01
   %   12:    8.332012e-02    8.376628e-01
   %   13:    7.099969e-02    8.521314e-01
   %   14:    6.145622e-02    8.655843e-01
   %   15:    5.394407e-02    8.777642e-01
   %   16:    4.792656e-02    8.884491e-01
   %   17:    4.301734e-02    8.975679e-01
   %   18:    3.893946e-02    9.052039e-01
   %   19:    3.549501e-02 

In [5]:

solver:set_preconditioner(ILU())
util.modsim2.SolverTest(domainDisc, approxSpace, solver, "SmootherError_ILU.vtk");


   % %%%%%%%%  Iterative Linear Solver  %%%%%%%%%%%%%%%%%%%
   % %%%%%%%%   (Precond: ILU)          %%%%%%%%%%%%%%%%%%%
   %   Iter      Defect         Rate 
   %    0:    3.835345e+01      -------
   %    1:    3.431112e+00    8.946031e-02
   %    2:    4.927598e-01    1.436152e-01
   %    3:    1.230239e-01    2.496629e-01
   %    4:    6.447890e-02    5.241171e-01
   %    5:    4.659528e-02    7.226438e-01
   %    6:    3.706219e-02    7.954065e-01
   %    7:    3.072135e-02    8.289135e-01
   %    8:    2.602688e-02    8.471920e-01
   %    9:    2.233216e-02    8.580421e-01
   %   10:    1.931522e-02    8.649059e-01
   %   11:    1.679363e-02    8.694508e-01
   %   12:    1.465328e-02    8.725501e-01
   %   13:    1.281723e-02    8.746999e-01
   %   14:    1.123051e-02    8.762043e-01
   %   15:    9.852091e-03    8.772614e-01
   %   16:    8.650188e-03    8.780053e-01
   %   17:    7.599443e-03    8.785292e-01
   %   18:    6.679135e-03    8.788980e-01
   %   19:    5.872013e-03 

## Mehrgitterverfahren

Obwohl die Konvergenzrate des Zweigitterverfahrens $h$-unabhängig ist, limitiert letztlich die Größe des  Grobgitterproblems immer noch die Komplexität. Ersetzt man dies durch eine rekursive Approximation, ergibt sich das **Mehrgitterverfahren**:

In [10]:
local myBaseLevel = 1
-- local smoother = SymmetricGaussSeidel()
local smoother = ILU()
local nu =1
-- Konstruktion (s.o., aber beachte baseLevel!)
mgv = GeometricMultiGrid(approxSpace)  -- Konstruktor
mgv:set_discretization(domainDisc)
mgv:set_base_level(0)          -- Mehrgitterverfahren!

-- Konfiguration des Glaetters
mgv:set_smoother(smoother)                  
mgv:set_num_presmooth(nu)  -- Vorglaettung
mgv:set_num_postsmooth(nu)  -- Nachglaettung
mgv:set_rap(true)          -- fuer Galerkinprodukt A_H=RAP, if true (alternative: assemble coarse system)

-- Konfiguration Grobgitterloeser
baseSolver = LU()
mgv:set_base_level(myBaseLevel)          -- Zweigitterverfahren
mgv:set_base_solver(baseSolver)     

mgv:set_cycle_type("V")       -- Select cycle type "V,W,F".


-- dbgWriter = GridFunctionDebugWriter(approxSpace)
-- mgv:set_debug(dbgWriter) -- REQ?
solver:set_preconditioner(mgv)

Auch hiermit können wir jetzt testen:

In [11]:
historyMGV = {}
configDesc.problem.parameters.epsy =1 
for nu=1,6 do
    print("1/eps="..1/configDesc.problem.parameters.epsy )
  
    local domainDisc=util.modsim2.CreateDiscretization(approxSpace,configDesc)
    historyMGV[nu] = util.modsim2.SolverTest(domainDisc, approxSpace, solver, "ErrorMGV")
    print()
   
    print()
    print("=========")
    configDesc.problem.parameters.epsy = configDesc.problem.parameters.epsy/10
end

1/eps=1
epsx=1
epsy=1

   % %%%%%%%%      Iterative Linear Solver      %%%%%%%%%%%
   % %%%%%%%%   (Precond: Geometric MultiGrid)  %%%%%%%%%%%
   %   Iter      Defect         Rate 
   %    0:    3.966045e+01      -------
   %    1:    3.914402e-01    9.869789e-03
   %    2:    1.380762e-02    3.527389e-02
   %    3:    8.138229e-04    5.894012e-02
   %    4:    5.853642e-05    7.192771e-02
   %    5:    4.555450e-06    7.782249e-02
   % Relative reduction 1.000000e-06 reached after 5 steps.
   % Average reduction over 5 steps: 4.092934e-02
   % %%%%%  Iteration converged  %%%%%

Results: 5 steps (rho=0.040929339978329)
Time: 0.115028 seconds


1/eps=10
epsx=1
epsy=0.1

   % %%%%%%%%      Iterative Linear Solver      %%%%%%%%%%%
   % %%%%%%%%   (Precond: Geometric MultiGrid)  %%%%%%%%%%%
   %   Iter      Defect         Rate 
   %    0:    2.409221e+01      -------
   %    1:    1.240787e-01    5.150160e-03
   %    2:    1.542692e-02    1.243317e-01
   %    3:    2.361425e-03    1.530717

In [9]:
dataTab = util.modsim2.CreateGnuplotDataTable(historyMGV)
plots = {
   { label = "MGV, eps=1",  data = dataTab[1], style = "linespoints", 1, 2}, 
   { label = "eps=10^-1",  data = dataTab[2], style = "linespoints", 1, 2}, 
   { label = "eps=10^-2",  data = dataTab[3], style = "linespoints", 1, 2}, 
   { label = "eps=10^-3",  data = dataTab[4], style = "linespoints", 1, 2}, 
   { label = "eps=10^-4",  data = dataTab[4], style = "linespoints", 1, 2}, 
   { label = "eps=10^-5",  data = dataTab[4], style = "linespoints", 1, 2}, 
}

options = {
    title = "Konvergenz MGV", 
    label = {x = "Iteration k", y = "Defekt |b-Ax^k|"},
    logscale = {x = false, y = true},
    terminal = "x11"
}

gnuplot.plot("Konvergenz_MGV.png", plots, options)

1
2
3
4
5
6
